In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

PROJECT_ROOT = Path.cwd().parent

DATASET_DIR = (
    PROJECT_ROOT
    / "dataset"
    / "CIC-IDS2017"
    / "MachineLearningCVE"
)

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATASET_DIR.exists())
print("CSV files:", len(list(DATASET_DIR.glob("*.csv"))))
print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)

Project root: C:\atharv\college\studies\coding\project a new start\NIDS
Dataset exists: True
CSV files: 8
Pandas version: 3.1.0
Scikit-learn version: 1.9.1


In [3]:
# Find all dataset files
csv_files = sorted(DATASET_DIR.glob("*.csv"))

# Inspect the first file without loading the entire dataset
df_sample = pd.read_csv(csv_files[0], nrows=5, low_memory=False)

# Clean whitespace around column names
df_sample.columns = df_sample.columns.str.strip()

print("File:", csv_files[0].name)
print("Sample shape:", df_sample.shape)

print("\nColumn names:")
print(df_sample.columns.tolist())

print("\nSample records:")
display(df_sample.head())

print("\nData types:")
display(df_sample.dtypes.value_counts())

File: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Sample shape: (5, 79)

Column names:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Cou

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN



Data types:


int64      74
float64     4
str         1
Name: count, dtype: int64

In [6]:
from collections import Counter

label_counts = Counter()

for file in csv_files:
    print("Reading:", file.name)

    # Read the original header without modifying its names
    header = pd.read_csv(file, nrows=0)

    # Find the original column name that corresponds to "Label"
    label_column = next(
        (
            col for col in header.columns
            if str(col).strip().lower() == "label"
        ),
        None,
    )

    if label_column is None:
        raise ValueError(
            f"Label column not found in {file.name}"
        )

    # Read only the label column in chunks
    for chunk in pd.read_csv(
        file,
        usecols=[label_column],
        chunksize=100_000,
        low_memory=False,
    ):
        labels = chunk[label_column].astype("string").str.strip()
        label_counts.update(labels.dropna().tolist())

label_summary = pd.DataFrame(
    label_counts.items(),
    columns=["Original Label", "Count"],
).sort_values("Count", ascending=False)

display(label_summary)

print("\nTotal rows:", label_summary["Count"].sum())
print("Unique original labels:", len(label_summary))

Reading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Reading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Reading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Reading: Monday-WorkingHours.pcap_ISCX.csv
Reading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Reading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Reading: Tuesday-WorkingHours.pcap_ISCX.csv
Reading: Wednesday-workingHours.pcap_ISCX.csv


,Original Label,Count
0,BENIGN,2273097
12,DoS Hulk,231073
2,PortScan,158930
1,DDoS,128027
13,DoS GoldenEye,10293
8,FTP-Patator,7938
9,SSH-Patator,5897
10,DoS slowloris,5796
11,DoS Slowhttptest,5499
3,Bot,1966



Total rows: 2830743
Unique original labels: 15


In [7]:
def map_label(label):
    label = str(label).strip()

    if label == "BENIGN":
        return "Normal"

    if label.startswith("DoS "):
        return "DoS"

    if label == "DDoS":
        return "DDoS"

    if label == "PortScan":
        return "PortScan"

    if label in {"FTP-Patator", "SSH-Patator"}:
        return "BruteForce"

    if label.startswith("Web Attack"):
        return "WebAttack"

    if label == "Bot":
        return "Bot"

    if label == "Infiltration":
        return "Infiltration"

    if label == "Heartbleed":
        return "Heartbleed"

    return "Unknown"


label_summary["Mapped Label"] = (
    label_summary["Original Label"].map(map_label)
)

display(
    label_summary[
        ["Original Label", "Mapped Label", "Count"]
    ]
)

print("\nMapped category totals:")
display(
    label_summary.groupby("Mapped Label")["Count"]
    .sum()
    .sort_values(ascending=False)
    .rename("Count")
    .to_frame()
)

print("\nUnknown labels:",
      int((label_summary["Mapped Label"] == "Unknown").sum()))

,Original Label,Mapped Label,Count
0,BENIGN,Normal,2273097
12,DoS Hulk,DoS,231073
2,PortScan,PortScan,158930
1,DDoS,DDoS,128027
13,DoS GoldenEye,DoS,10293
8,FTP-Patator,BruteForce,7938
9,SSH-Patator,BruteForce,5897
10,DoS slowloris,DoS,5796
11,DoS Slowhttptest,DoS,5499
3,Bot,Bot,1966



Mapped category totals:


,Count
Mapped Label,
Normal,2273097
DoS,252661
PortScan,158930
DDoS,128027
BruteForce,13835
WebAttack,2180
Bot,1966
Infiltration,36
Heartbleed,11



Unknown labels: 0


In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

CHUNK_SIZE = 100_000

def load_and_clean_csv(file_path):
    """Load one CSV in chunks and apply consistent basic cleaning."""
    chunks = []

    for chunk in pd.read_csv(
        file_path,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ):
        # Normalize column names
        chunk.columns = chunk.columns.str.strip()

        # Find the label column safely
        label_column = next(
            (
                col for col in chunk.columns
                if col.lower() == "label"
            ),
            None,
        )

        if label_column is None:
            raise ValueError(f"Label column missing in {file_path.name}")

        # Normalize labels and map to target categories
        chunk["Target"] = chunk[label_column].map(map_label)

        # Remove the original label column
        chunk = chunk.drop(columns=[label_column])

        # Convert feature columns to numeric
        feature_columns = [
            col for col in chunk.columns if col != "Target"
        ]

        chunk[feature_columns] = chunk[feature_columns].apply(
            pd.to_numeric, errors="coerce"
        )

        # Replace infinite values and remove rows with invalid features
        chunk[feature_columns] = chunk[feature_columns].replace(
            [np.inf, -np.inf], np.nan
        )

        chunk = chunk.dropna(
            subset=feature_columns + ["Target"]
        )

        chunks.append(chunk)

    return pd.concat(chunks, ignore_index=True)


print("Dataset loader defined successfully.")

Dataset loader defined successfully.


In [9]:
test_df = load_and_clean_csv(csv_files[0])

print("File:", csv_files[0].name)
print("Cleaned shape:", test_df.shape)

print("\nTarget distribution:")
display(test_df["Target"].value_counts().to_frame("Count"))

print("\nMissing values:", int(test_df.isna().sum().sum()))
print("Infinite values:", int(np.isinf(
    test_df.select_dtypes(include=np.number).to_numpy()
).sum()))

print("\nFirst five rows:")
display(test_df.head())

File: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Cleaned shape: (225711, 79)

Target distribution:


,Count
Target,
DDoS,128025
Normal,97686



Missing values: 0
Infinite values: 0

First five rows:


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Target
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal


In [10]:
print("Test dataset memory usage:")
print(f"{test_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

import psutil

memory = psutil.virtual_memory()

print(f"\nTotal RAM: {memory.total / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")

Test dataset memory usage:
145.91 MB

Total RAM: 15.73 GB
Available RAM: 1.95 GB


In [11]:
import gc
import psutil

del test_df
gc.collect()

memory = psutil.virtual_memory()

print(f"Available RAM: {memory.available / (1024**3):.2f} GB")

Available RAM: 1.86 GB
